# Adım 3: Spark Structured Streaming ile Veri Okuma

Bu notebook, Kafka'dan gelen streaming veriyi **Bronze → Silver → Gold** katmanlarına işleyen Spark pipeline'ını açıklar ve test eder.

| Katman | Konum | İçerik |
|--------|-------|--------|
| **Bronze** | `data/delta/bronze/` | Ham JSON, kafka metadata, sha256 |
| **Silver** | `data/delta/silver/` | Temizlenmiş, zenginleştirilmiş trip verisi |
| **Gold** | `data/delta/gold/` | ML için hazır feature seti |

## 1. Konfigürasyon

In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = Path(os.getcwd()).parent

CONFIG = {
    "KAFKA_BOOTSTRAP": os.getenv("KAFKA_BOOTSTRAP_SERVERS", "localhost:29092"),
    "KAFKA_TOPIC":     os.getenv("KAFKA_TOPIC", "yellow_taxi_trips"),
    "BRONZE_PATH":     str(PROJECT_ROOT / "data/delta/bronze/yellow_taxi_trips"),
    "SILVER_PATH":     str(PROJECT_ROOT / "data/delta/silver/yellow_taxi_trips"),
    "GOLD_PATH":       str(PROJECT_ROOT / "data/delta/gold/fare_features"),
    "CHECKPOINT":      str(PROJECT_ROOT / "data/delta/checkpoints/bronze_yellow_taxi_trips"),
    "LOOKUP_PATH":     str(PROJECT_ROOT / "data/lookup/taxi_zone_lookup.csv"),
}

print("=== Pipeline Konfigürasyonu ===")
for k, v in CONFIG.items():
    print(f"  {k:<20} = {v}")

## 2. Spark Job Dosyaları

In [ ]:
spark_jobs = [
    "stream_to_bronze.py",
    "bronze_to_silver.py",
    "silver_to_gold.py",
    "schemas.py",
    "common.py",
]

print("spark_jobs/ dosya kontrolü:")
print("-" * 45)
for f in spark_jobs:
    yol = PROJECT_ROOT / "spark_jobs" / f
    boyut = yol.stat().st_size if yol.exists() else 0
    icon = "✅" if yol.exists() else "❌"
    print(f"  {icon} {f:<30} {boyut:>6} byte")

## 3. JSON Şema Tanımı (`schemas.py`)

Kafka mesajlarındaki JSON'u parse etmek için kullanılan PySpark şeması:

In [ ]:
YELLOW_TAXI_FIELDS = [
    ("VendorID",               "IntegerType",  "Taksi firması (1=CMT, 2=VTS)"),
    ("tpep_pickup_datetime",   "StringType",   "Biniş zamanı"),
    ("tpep_dropoff_datetime",  "StringType",   "İniş zamanı"),
    ("passenger_count",        "DoubleType",   "Yolcu sayısı"),
    ("trip_distance",          "DoubleType",   "Mesafe (mil)"),
    ("RatecodeID",             "DoubleType",   "Ücret kodu"),
    ("store_and_fwd_flag",     "StringType",   "Kayıt modu"),
    ("PULocationID",           "IntegerType",  "Biniş bölgesi"),
    ("DOLocationID",           "IntegerType",  "İniş bölgesi"),
    ("payment_type",           "IntegerType",  "Ödeme tipi"),
    ("fare_amount",            "DoubleType",   "Ücret ($)"),
    ("extra",                  "DoubleType",   "Ek ücret"),
    ("mta_tax",                "DoubleType",   "MTA vergisi"),
    ("tip_amount",             "DoubleType",   "Bahşiş ($)"),
    ("tolls_amount",           "DoubleType",   "Köprü/tünel ücreti"),
    ("improvement_surcharge",  "DoubleType",   "İyileştirme fonu"),
    ("total_amount",           "DoubleType",   "Toplam ücret ($)"),
    ("congestion_surcharge",   "DoubleType",   "Yoğunluk fonu"),
    ("airport_fee",            "DoubleType",   "Havaalanı ücreti"),
    ("cbd_congestion_fee",     "DoubleType",   "CBD yoğunluk ücreti"),
]

print(f"{'Alan Adı':<30} {'Tip':<15} {'Açıklama'}")
print("-" * 65)
for alan, tip, aciklama in YELLOW_TAXI_FIELDS:
    print(f"  {alan:<28} {tip:<15} {aciklama}")
print(f"\nToplam: {len(YELLOW_TAXI_FIELDS)} alan")

## 4. Adım 3a: Stream → Bronze (`stream_to_bronze.py`)

Kafka topic'inden ham JSON mesajları okunur, metadata eklenir ve Delta formatında yazılır:

In [ ]:
bronze_pipeline = """
Kafka Topic
    └─► stream_df (key, value, topic, partition, offset, timestamp)
         ├─ value → cast("string") → raw_json
         ├─ key   → cast("string") → message_key
         ├─ current_timestamp()   → ingested_at
         └─ sha2(raw_json, 256)   → raw_json_sha256
              │
              ▼
         FILTER: raw_json IS NOT NULL AND length > 0
              │
              ▼
         Bronze Delta Table
         data/delta/bronze/yellow_taxi_trips/
"""
print(bronze_pipeline)

# stream_to_bronze.py'nin kaynak kodunu göster
with open(PROJECT_ROOT / "spark_jobs" / "stream_to_bronze.py", "r") as f:
    print(f.read())

## 5. Adım 3b: Bronze → Silver (`bronze_to_silver.py`)

Ham JSON parse edilir, temizlenir ve zone bilgisiyle zenginleştirilir:

In [ ]:
silver_pipeline = """
Bronze Delta
    └─► parse_bronze_records()
         └─ from_json(raw_json, YELLOW_TAXI_SCHEMA) → trip.*
              │
              ▼
         clean_trips()
         ├─ pickup_datetime / dropoff_datetime  (timestamp parse)
         ├─ trip_duration_minutes               (türetilmiş)
         ├─ pickup_date / pickup_hour / pickup_day_of_week
         └─ FILTER: mesafe 0-100 mil, ücret 0-500$, süre 0-240dk
              │
              ▼
         enrich_with_zones()
         ├─ JOIN taxi_zone_lookup.csv → pickup_borough/zone
         └─ JOIN taxi_zone_lookup.csv → dropoff_borough/zone
              │
              ▼
         Silver Delta Table (partition: pickup_year / pickup_month)
         data/delta/silver/yellow_taxi_trips/
"""
print(silver_pipeline)

### Temizleme Kuralları

In [ ]:
kurallar = [
    ("pickup_datetime",      "IS NOT NULL",    "Geçersiz biniş zamanı"),
    ("dropoff_datetime",     "IS NOT NULL",    "Geçersiz iniş zamanı"),
    ("dropoff > pickup",     "True",           "Negatif süre tespiti"),
    ("trip_distance",        "0 < d <= 100",   "Anlamsız mesafe filtreleme"),
    ("fare_amount",          "0 < f <= 500",   "Anlamsız ücret filtreleme"),
    ("trip_duration_minutes","0 < t <= 240",   "Max 4 saat süre"),
    ("PULocationID",         "IS NOT NULL",    "Biniş bölgesi zorunlu"),
    ("DOLocationID",         "IS NOT NULL",    "İniş bölgesi zorunlu"),
    ("passenger_count",      "0-8 veya NULL",  "NULL kabul edilir"),
]

print(f"{'Alan':<30} {'Kural':<18} {'Gerekçe'}")
print("-" * 70)
for alan, kural, gerekcesi in kurallar:
    print(f"  {alan:<28} {kural:<18} {gerekcesi}")

## 6. Adım 3c: Silver → Gold (`silver_to_gold.py`)

ML modeli için feature seçimi ve türetme:

In [ ]:
gold_features = [
    ("label",               "fare_amount",         "Hedef değişken (tahmin edilecek)"),
    ("trip_distance",       "Silver'dan",           "Yolculuk mesafesi"),
    ("passenger_count",     "Silver'dan",           "Yolcu sayısı"),
    ("pickup_hour",         "Silver'dan",           "Saat bilgisi"),
    ("pickup_day_of_week",  "Silver'dan",           "Haftanın günü"),
    ("PULocationID",        "Silver'dan",           "Biniş bölgesi ID"),
    ("DOLocationID",        "Silver'dan",           "İniş bölgesi ID"),
    ("RatecodeID",          "Silver'dan",           "Ücret kodu"),
    ("payment_type",        "Silver'dan",           "Ödeme tipi"),
    ("pickup_borough",      "Silver'dan",           "Biniş ilçesi"),
    ("dropoff_borough",     "Silver'dan",           "İniş ilçesi"),
    ("is_airport_trip",     "TÜRETILMIŞ (0/1)",     "Havaalanı yolculuğu flag"),
]

print(f"{'Feature':<25} {'Kaynak':<22} {'Açıklama'}")
print("-" * 72)
for feat, kaynak, aciklama in gold_features:
    print(f"  {feat:<23} {kaynak:<22} {aciklama}")
print(f"\nToplam: {len(gold_features)} feature (1 label + {len(gold_features)-1} özellik)")

## 7. Spark Job'larını Docker ile Çalıştırma

In [ ]:
import subprocess, os
os.chdir(PROJECT_ROOT)

# Bronze job
bronze_cmd = (
    "docker exec nyc-taxi-spark "
    "spark-submit --master local[*] "
    "--packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0,"
    "io.delta:delta-spark_2.12:3.1.0 "
    "--conf spark.sql.extensions=io.delta.sql.DeltaSparkSessionExtension "
    "--conf spark.sql.catalog.spark_catalog=org.apache.spark.sql.delta.catalog.DeltaCatalog "
    "/app/spark_jobs/stream_to_bronze.py"
)

silver_cmd = bronze_cmd.replace(
    "--packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0,io.delta:delta-spark_2.12:3.1.0 ",
    "--packages io.delta:delta-spark_2.12:3.1.0 "
).replace("stream_to_bronze.py", "bronze_to_silver.py")

gold_cmd = silver_cmd.replace("bronze_to_silver.py", "silver_to_gold.py")

print("=== Çalıştırma Komutları ===")
print("\n[1] Bronze (Kafka → Delta):")
print(f"  {bronze_cmd[:120]}...")
print("\n[2] Silver (Bronze → Temiz Delta):")
print(f"  {silver_cmd[:120]}...")
print("\n[3] Gold (Silver → ML Features):")
print(f"  {gold_cmd[:120]}...")

### 7a. Bronze Job'ını Başlat

In [ ]:
# Bronze job'ı çalıştır
print("Bronze streaming job başlatılıyor...")
result = subprocess.run(bronze_cmd, shell=True, capture_output=True, text=True, timeout=300)
print(result.stdout[-3000:] if result.stdout else "")
if result.stderr:
    # Sadece WARNING/ERROR satırlarını filtrele
    hatalar = [l for l in result.stderr.splitlines() if any(x in l for x in ["ERROR", "Exception", "WARN"])]
    if hatalar:
        print("\nDikkat:")
        for h in hatalar[-10:]:
            print(f"  {h}")
print(f"\nReturncode: {result.returncode}")

### 7b. Silver Job'ını Başlat

In [ ]:
print("Silver job başlatılıyor...")
result = subprocess.run(silver_cmd, shell=True, capture_output=True, text=True, timeout=300)
print(result.stdout[-3000:] if result.stdout else "")
print(f"Returncode: {result.returncode}")

### 7c. Gold Job'ını Başlat

In [ ]:
print("Gold job başlatılıyor...")
result = subprocess.run(gold_cmd, shell=True, capture_output=True, text=True, timeout=300)
print(result.stdout[-3000:] if result.stdout else "")
print(f"Returncode: {result.returncode}")

## 8. Delta Katman Kontrolü

In [ ]:
import os

katmanlar = [
    ("Bronze", CONFIG["BRONZE_PATH"]),
    ("Silver", CONFIG["SILVER_PATH"]),
    ("Gold",   CONFIG["GOLD_PATH"]),
]

print(f"{'Katman':<10} {'Durum':<10} {'Klasör'}")
print("-" * 60)
for ad, yol in katmanlar:
    delta_log = Path(yol) / "_delta_log"
    if delta_log.exists():
        dosya_sayisi = len(list(Path(yol).rglob("*.parquet")))
        print(f"  ✅ {ad:<8} {dosya_sayisi} parquet dosyası  {yol}")
    elif Path(yol).exists():
        print(f"  ⚠️  {ad:<8} Delta log yok      {yol}")
    else:
        print(f"  ❌ {ad:<8} Klasör yok         {yol}")

## 9. Özet & Teslim Edilecekler

| # | Teslim | Durum |
|---|--------|-------|
| 1 | `stream_to_bronze.py` — Kafka → Bronze Delta | ✅ |
| 2 | `bronze_to_silver.py` — JSON parse, temizleme, zone zenginleştirme | ✅ |
| 3 | `silver_to_gold.py` — ML feature seçimi | ✅ |
| 4 | `schemas.py` — YELLOW_TAXI_SCHEMA tanımı | ✅ |
| 5 | Null/aykırı değer filtreleme kuralları | ✅ (Bölüm 5) |

### Delta Katman Mimarisi
```
Kafka ──► Bronze (ham) ──► Silver (temiz+zengin) ──► Gold (ML features)
```

---
> **Sonraki adım →** Adım 4: Keşifsel Veri Analizi (EDA)